In [2]:
!pip install pymongo

   ---------------------------------------- 0.0/920.3 kB ? eta -:--:--
   ---------------------------------------- 10.2/920.3 kB ? eta -:--:--
   - ------------------------------------- 30.7/920.3 kB 259.2 kB/s eta 0:00:04
   - ------------------------------------- 41.0/920.3 kB 326.8 kB/s eta 0:00:03
   - ------------------------------------- 41.0/920.3 kB 326.8 kB/s eta 0:00:03
   - ------------------------------------- 41.0/920.3 kB 326.8 kB/s eta 0:00:03
   ---- --------------------------------- 112.6/920.3 kB 409.6 kB/s eta 0:00:02
   ---- --------------------------------- 112.6/920.3 kB 409.6 kB/s eta 0:00:02
   ------- ------------------------------ 174.1/920.3 kB 476.3 kB/s eta 0:00:02
   ------- ------------------------------ 174.1/920.3 kB 476.3 kB/s eta 0:00:02
   -------- ----------------------------- 204.8/920.3 kB 428.5 kB/s eta 0:00:02
   ------------ ------------------------- 307.2/920.3 kB 633.2 kB/s eta 0:00:01
   --------------- ---------------------- 368.6/920.3 kB 

In [ ]:
import pymongo
from datetime import datetime

class StockManagementSystem:
    def __init__(self, connection_string="mongodb://localhost:27017/", db_name="inventory_db"):
        self.client = pymongo.MongoClient(connection_string)
        self.db = self.client[db_name]
        self.products = self.db["products"]
        self.products.create_index("sku", unique=True)
        print("Connected to MongoDB successfully!")

    def add_product(self, sku, name, category, price, quantity):
        product = {
            "sku": sku,
            "name": name,
            "category": category,
            "price": price,
            "quantity": quantity,
            "last_updated": datetime.now()
        }
        try:
            self.products.insert_one(product)
            print(f"Product '{name}' added successfully.")
        except pymongo.errors.DuplicateKeyError:
            print(f"Error: Product with SKU '{sku}' already exists!")

    def view_all_products(self):
        print("\n--- Current Inventory ---")
        items = self.products.find()
        for item in items:
            print(f"SKU: {item['sku']} | Name: {item['name']} | Stock: {item['quantity']} | Price: ${item['price']}")
        print("-------------------------\n")

    def find_product(self, sku):
        product = self.products.find_one({"sku": sku})
        if product:
            print(f"Found: {product['name']} - {product['quantity']} in stock.")
            return product
        else:
            print(f"Product with SKU '{sku}' not found.")
            return None

    def update_stock(self, sku, quantity_change):
        product = self.products.find_one({"sku": sku})
        if not product:
            print(f"Product with SKU '{sku}' not found.")
            return

        new_quantity = product['quantity'] + quantity_change
        
        if new_quantity < 0:
            print(f"Cannot remove {abs(quantity_change)} items. Only {product['quantity']} in stock.")
            return

        self.products.update_one(
            {"sku": sku},
            {"$set": {
                "quantity": new_quantity,
                "last_updated": datetime.now()
            }}
        )
        print(f"Stock updated for '{product['name']}'. New quantity: {new_quantity}")

    def delete_product(self, sku):
        result = self.products.delete_one({"sku": sku})
        if result.deleted_count > 0:
            print(f"Product with SKU '{sku}' deleted.")
        else:
            print(f"Product with SKU '{sku}' not found.")

if __name__ == "__main__":
    sms = StockManagementSystem()
    
    sms.add_product("LAP-001", "Dell XPS 15", "Electronics", 1500.00, 10)
    sms.add_product("MOU-002", "Logitech MX Master 3", "Accessories", 99.99, 50)
    
    sms.view_all_products()
    
    sms.update_stock("LAP-001", 5)
    sms.update_stock("MOU-002", -2)
    sms.update_stock("LAP-001", -20)
    
    sms.view_all_products()
    
    sms.delete_product("MOU-002")